# Подготовка

## Импорты

In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

from src.dataset import DatasetStore
from src.runner import ExperimentRunner
from src.pipeline import Pipeline
from src.preprocessors import *
from src.models import *
from src.validators import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Сохранение в ONNX

In [3]:
rf = ExperimentRunner.load_pipeline('_analys/RandomForestTop15/pipeline.joblib')
rf.export_onnx(
    path="_export",
)

WindowsPath('_export/model.onnx')

# Использование ONNX

## Импорты

In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

from src.dataset import DatasetStore
from src.runner import ExperimentRunner
from src.pipeline import Pipeline
from src.preprocessors import *
from src.models import *
from src.validators import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [3]:
df, schema = DatasetStore.load(
    "data/processed/dataset.csv",
    "data/processed/dataset.schema.json",
    include_missing=False,
)
feature_info = DatasetStore.get_feature_info(
    schema,
    include_missing=False,
)

In [4]:
columns = ['bled cut', 'albumin']
for col in columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [5]:
df["diagnosis"] = df["diagnosis"].replace(6, 5)

y = LabelEncoder().fit_transform(df['diagnosis'].values)
X = df.drop(columns=['diagnosis'])

class_labels = [0, 1]

## Инференс ONNX

In [6]:
import json
import numpy as np
import onnxruntime as ort

EXPORT_DIR = "_export"

with open(f"{EXPORT_DIR}/meta.json", encoding="utf-8") as f:
    meta = json.load(f)

FEATURES = [i["name"] for i in meta["model_inputs"]]   # порядок 15 фичей
CLASSES  = meta["classes"]                             # [0, 1]

sess = ort.InferenceSession(
    f"{EXPORT_DIR}/model.onnx",
    providers=["CPUExecutionProvider"],
)

# Полезно один раз посмотреть, что реально ждёт граф
for i in sess.get_inputs():
    print("IN :", i.name, i.shape, i.type)
for o in sess.get_outputs():
    print("OUT:", o.name, o.shape, o.type)

IN : meteorism [None, 1] tensor(string)
IN : LS_cen_gid [None, 1] tensor(string)
IN : rigid_of_abdomi [None, 1] tensor(string)
IN : Ur [None, 1] tensor(string)
IN : LS_steat [None, 1] tensor(string)
IN : glu [None, 1] tensor(string)
IN : intensiv [None, 1] tensor(string)
IN : pain_of_adbomin [None, 1] tensor(string)
IN : Ht [None, 1] tensor(string)
IN : pain [None, 1] tensor(string)
IN : amilase [None, 1] tensor(string)
IN : s__peritonit [None, 1] tensor(string)
IN : icterus [None, 1] tensor(string)
IN : mramor_cut [None, 1] tensor(string)
IN : bil [None, 1] tensor(string)
OUT: label [None] tensor(int64)
OUT: probabilities [None, 2] tensor(float)


In [15]:
import re
import numpy as np

def sanitize(name: str) -> str:
    """Правило skl2onnx: всё, что не [A-Za-z0-9_], → '_'."""
    return re.sub(r"[^A-Za-z0-9_]", "_", name)

TOP_15 = [
    "meteorism", "LS/cen gid", "rigid of abdomi", "Ur", "LS steat",
    "glu", "intensiv", "pain of adbomin", "Ht", "pain",
    "amilase", "s. peritonit", "icterus", "mramor cut", "bil",
]
CLASSES = [0, 1]

# Проверим, что все санитизированные имена реально есть среди входов модели
onnx_inputs = {i.name for i in sess.get_inputs()}
missing = [sanitize(f) for f in TOP_15 if sanitize(f) not in onnx_inputs]
assert not missing, f"Нет таких входов в графе: {missing}"


def predict_series(sess, x_row, features=TOP_15, classes=CLASSES):
    feeds = {}
    for orig in features:                      # ← ключ в Series — оригинал
        v = str(x_row[orig])
        col = np.empty((1, 1), dtype=object)
        col[0, 0] = v
        feeds[sanitize(orig)] = col            # ← ключ в feeds — санитизированный

    out_label, out_proba = sess.run(["label", "probabilities"], feeds)
    return np.asarray(out_proba, dtype=float).ravel()

In [8]:
top_15_features = [
    "meteorism",
    "LS/cen gid",
    "rigid of abdomi",
    "Ur",
    "LS steat",
    "glu",
    "intensiv",
    "pain of adbomin",
    "Ht",
    "pain",
    "amilase",
    "s. peritonit",
    "icterus",
    "mramor cut",
    "bil",
]
X_new = df[top_15_features]

In [16]:
proba_onnx = predict_series(sess, X_new.iloc[100])
proba_onnx

array([0.64217579, 0.35782421])

## Инференс pipeline

In [14]:
rf = ExperimentRunner.load_pipeline('_analys/RandomForestTop15/pipeline.joblib')
proba_pp = rf.predict_proba(X_new)[100]
proba_pp

array([0.64217579, 0.35782421])

## Сравнение вероятностей

In [18]:
np.allclose(proba_onnx, proba_pp)

True